# 🔁 Lección 3 – Elementos Básicos de Spark: RDD, Transformaciones y Acciones
### Proyecto: Retail Analytics Pipeline — RetailMax
**Módulo 9: Fundamentos de Big Data | Alkemy**

---
**Objetivo:** Manipular grandes volúmenes de datos mediante RDDs aplicando transformaciones lazy y acciones que disparan la ejecución.

In [ ]:
# ============================================================
# Setup: SparkSession + carga del dataset
# ============================================================
from pyspark.sql import SparkSession
import os, numpy as np, matplotlib.pyplot as plt

spark = (
    SparkSession.builder
    .appName('RetailMax_L3_RDD')
    .master('local[*]')
    .config('spark.driver.memory', '2g')
    .config('spark.ui.showConsoleProgress', 'false')
    .getOrCreate()
)
sc = spark.sparkContext
sc.setLogLevel('WARN')

DATA_PATH = '../data/fashion_mnist'

def cargar_rdd(sc, filepath):
    """Carga un CSV de Fashion-MNIST en un RDD tipado."""
    rdd_raw = sc.textFile(filepath)
    header = rdd_raw.first()
    return (
        rdd_raw
        .filter(lambda line: line != header)
        .map(lambda line: line.split(','))
        .map(lambda f: (int(f[0]), f[1], [int(x) for x in f[2:]]))
    )

rdd_train = cargar_rdd(sc, os.path.join(DATA_PATH, 'fashion_train.csv'))
rdd_test  = cargar_rdd(sc, os.path.join(DATA_PATH, 'fashion_test.csv'))

rdd_train.cache()
rdd_test.cache()
print(f'✅ RDD train: {rdd_train.count():,} registros | test: {rdd_test.count():,}')

## 1. Transformaciones vs Acciones

| Tipo | Descripción | Ejemplos | ¿Cuándo ejecuta? |
|------|-------------|----------|------------------|
| **Transformación** | Genera un nuevo RDD (lazy) | `map`, `filter`, `flatMap`, `distinct`, `sortBy` | Solo al llamar una acción |
| **Acción** | Devuelve resultado al driver | `collect`, `count`, `take`, `sum`, `mean` | Inmediatamente |

> 🔑 **Ejecución lazy:** Las transformaciones **no se ejecutan** hasta que se llama una acción. Spark construye un DAG (Directed Acyclic Graph) y optimiza el plan antes de ejecutar.

In [ ]:
# ============================================================
# TRANSFORMACIÓN 1: map() — extraer label y primer pixel
# ============================================================
# map: aplica una función a cada elemento → 1 entrada, 1 salida
rdd_label_pixel = rdd_train.map(
    lambda r: (r[0], r[1], r[2][0])  # (label_id, label_name, pixel_0)
)

print('=== map() — Extraer label y pixel_0 ===')
print('Primeros 5 elementos:')
for item in rdd_label_pixel.take(5):
    print(f'  {item}')

In [ ]:
# ============================================================
# TRANSFORMACIÓN 2: filter() — solo prendas superiores
# ============================================================
# Clases de 'parte superior': T-shirt(0), Pullover(2), Coat(4), Shirt(6)
UPPER_BODY = {'T-shirt/top', 'Pullover', 'Coat', 'Shirt'}

rdd_upper = rdd_train.filter(lambda r: r[1] in UPPER_BODY)

count_upper = rdd_upper.count()
print(f'=== filter() — Prendas de parte superior ===')
print(f'Total registros  : {rdd_train.count():,}')
print(f'Prendas superiores: {count_upper:,} ({count_upper/rdd_train.count()*100:.1f}%)')

In [ ]:
# ============================================================
# TRANSFORMACIÓN 3: flatMap() — expandir cada pixel como fila
# ============================================================
# flatMap: 1 entrada → N salidas (aplana el resultado)
# Ejemplo: obtener todos los valores de los primeros 5 píxeles
rdd_pixels_flat = (
    rdd_train
    .map(lambda r: (r[0], r[2][:5]))       # Solo primeros 5 píxeles por muestra
    .flatMap(lambda r: [(r[0], px) for px in r[1]])  # Explode: (label, pixel_val)
)

print('=== flatMap() — Expandir píxeles ===')
print('Ejemplo (label_id, pixel_value):')
print(rdd_pixels_flat.take(10))
print(f'Total pares (label, pixel): {rdd_pixels_flat.count():,}')

In [ ]:
# ============================================================
# TRANSFORMACIÓN 4: distinct() — clases únicas
# ============================================================
rdd_clases = rdd_train.map(lambda r: r[1]).distinct().sortBy(lambda x: x)

print('=== distinct() + sortBy() — Clases únicas ordenadas ===')
print(rdd_clases.collect())

In [ ]:
# ============================================================
# TRANSFORMACIÓN 5: sortBy() — ordenar por conteo de clase
# ============================================================
rdd_conteo_clases = (
    rdd_train
    .map(lambda r: (r[1], 1))
    .reduceByKey(lambda a, b: a + b)
    .sortBy(lambda x: -x[1])    # Ordenar descendente por conteo
)

print('=== sortBy() — Clases ordenadas por frecuencia (DESC) ===')
for nombre, cnt in rdd_conteo_clases.collect():
    print(f'  {nombre:<15} : {cnt:>6,}')

In [ ]:
# ============================================================
# Pair RDDs — clave-valor para agregaciones por categoría
# ============================================================
# Pair RDD: (key, value) — habilita reduceByKey, groupByKey, etc.

# Pair RDD: (label_name, pixels)
rdd_pair = rdd_train.map(lambda r: (r[1], r[2]))

# Calcular la media del píxel central (pixel 391 ≈ centro de 28x28) por clase
CENTRO = 391
rdd_media_centro = (
    rdd_train
    .map(lambda r: (r[1], (r[2][CENTRO], 1)))          # (clase, (pixel_val, 1))
    .reduceByKey(lambda a, b: (a[0]+b[0], a[1]+b[1]))  # Suma acumulada
    .map(lambda r: (r[0], round(r[1][0] / r[1][1], 2))) # Promedio
    .sortBy(lambda x: -x[1])
)

print('=== Pair RDD — Media del píxel central por categoría ===')
print('(Mayor valor = más claro/activo en el centro de la prenda)')
for clase, media in rdd_media_centro.collect():
    bar = '█' * int(media / 10)
    print(f'  {clase:<15}: {media:>6.2f}  {bar}')

In [ ]:
# ============================================================
# ACCIONES: collect, sum, mean, stdev sobre intensidades
# ============================================================
# Extraer intensidad media por imagen (promedio de 784 píxeles)
rdd_intensidad = rdd_train.map(
    lambda r: (r[1], sum(r[2]) / len(r[2]))  # (clase, intensidad_media)
)

# Solo los valores numéricos para estadísticas globales
rdd_vals = rdd_intensidad.map(lambda r: r[1])

print('=== ACCIONES: Estadísticas globales de intensidad media ===')
print(f'count() : {rdd_vals.count():,}')
print(f'sum()   : {rdd_vals.sum():.2f}')
print(f'mean()  : {rdd_vals.mean():.4f}')
print(f'stdev() : {rdd_vals.stdev():.4f}')
print(f'min()   : {rdd_vals.min():.4f}')
print(f'max()   : {rdd_vals.max():.4f}')

In [ ]:
# ============================================================
# Documentar el LINAJE del RDD
# ============================================================
# El linaje muestra el DAG de transformaciones aplicadas
print('=== LINAJE DEL RDD (toDebugString) ===')
print(rdd_intensidad.toDebugString().decode('utf-8'))

In [ ]:
# ============================================================
# Estadísticas por clase — acciones sobre Pair RDDs
# ============================================================
from pyspark.rdd import portable_hash

# Media de intensidad por categoría
rdd_stats_clase = (
    rdd_intensidad
    .map(lambda r: (r[0], (r[1], r[1]**2, 1)))    # (clase, (val, val^2, count))
    .reduceByKey(lambda a, b: (a[0]+b[0], a[1]+b[1], a[2]+b[2]))
    .map(lambda r: (
        r[0],
        round(r[1][0] / r[1][2], 4),                           # media
        round((r[1][1]/r[1][2] - (r[1][0]/r[1][2])**2)**0.5, 4)  # stdev
    ))
    .sortBy(lambda x: -x[1])
)

print('=== Intensidad media y stdev por categoría ===')
print(f'{"Categoría":<15} | {"Media":>8} | {"Stdev":>8}')
print('-' * 38)
for clase, media, stdev in rdd_stats_clase.collect():
    print(f'{clase:<15} | {media:>8.4f} | {stdev:>8.4f}')

In [ ]:
# ============================================================
# Visualización: Intensidad media por categoría
# ============================================================
stats = rdd_stats_clase.collect()
clases = [s[0] for s in stats]
medias = [s[1] for s in stats]
stdevs = [s[2] for s in stats]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Gráfico de barras con error
colors = plt.cm.viridis(np.linspace(0.2, 0.9, len(clases)))
bars = axes[0].barh(clases, medias, xerr=stdevs, color=colors, capsize=4)
axes[0].set_xlabel('Intensidad media (0-255)')
axes[0].set_title('Intensidad Promedio de Píxeles por Categoría\n(con desviación estándar)', fontweight='bold')
axes[0].axvline(x=sum(medias)/len(medias), color='red', linestyle='--', label='Media global')
axes[0].legend()

# Scatterplot media vs stdev
axes[1].scatter(medias, stdevs, c=colors, s=100, zorder=5)
for i, clase in enumerate(clases):
    axes[1].annotate(clase, (medias[i], stdevs[i]), fontsize=7, 
                     xytext=(3, 3), textcoords='offset points')
axes[1].set_xlabel('Intensidad media')
axes[1].set_ylabel('Desviación estándar')
axes[1].set_title('Media vs Variabilidad por Categoría', fontweight='bold')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
os.makedirs('../visualizations', exist_ok=True)
plt.savefig('../visualizations/L3_intensidad_por_categoria.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Visualización guardada: visualizations/L3_intensidad_por_categoria.png')

In [ ]:
# ============================================================
# cache() vs persist() — optimización
# ============================================================
from pyspark import StorageLevel

# cache() = persist(MEMORY_ONLY) — almacena en RAM
rdd_intensidad.cache()

# persist() permite elegir el nivel de almacenamiento
# Opciones: MEMORY_ONLY, MEMORY_AND_DISK, DISK_ONLY, OFF_HEAP
rdd_pair.persist(StorageLevel.MEMORY_AND_DISK)

print('=== OPTIMIZACIÓN: cache() y persist() ===')
print('rdd_intensidad → cache()                  (MEMORY_ONLY)')
print('rdd_pair       → persist(MEMORY_AND_DISK) (RAM + disco si no cabe en memoria)')
print('\n💡 Usar cache/persist cuando un RDD se reutiliza múltiples veces')
print('   Evitar en RDDs de un solo uso para ahorrar recursos.')

## Resumen del DAG generado

```
sc.textFile(CSV)
  └─ filter(header)
      └─ map(split ',')
          └─ map(parse types)           ← rdd_train [CACHED]
              ├─ map(label, pixels)     ← rdd_pair
              │   └─ reduceByKey(...)   ← estadísticas
              └─ map(label, intensity)  ← rdd_intensidad [CACHED]
                  ├─ count()  → 60000
                  ├─ mean()   → estadística global
                  └─ stdev()  → estadística global
```

> ⚡ Las transformaciones son **lazy**: el DAG completo se construye en memoria sin ejecutar hasta que se llama una acción (`count`, `collect`, `mean`, etc.).

---
## ✅ Checklist Lección 3
- [x] RDDs creados a partir de datos Fashion-MNIST
- [x] Pair RDDs creados y utilizados para agregaciones por clase
- [x] Transformaciones aplicadas: `map`, `filter`, `flatMap`, `distinct`, `sortBy`, `reduceByKey`
- [x] Acciones ejecutadas: `collect`, `count`, `sum`, `mean`, `stdev`, `min`, `max`
- [x] Linaje documentado con `toDebugString()`
- [x] Optimización con `cache()` y `persist()`
- [x] Visualizaciones generadas y guardadas

**Próximo paso → Lección 4:** DataFrames, Spark SQL y generación de métricas de negocio.